# UAV Computer Vision Project

## Phase 4: Data Preprocessing

### Objectives

- Convert VisDrone annotations to YOLO format
- Create processed dataset
- Verify label conversion
- Prepare dataset for YOLOv11 training

In [32]:
import cv2
import shutil

from pathlib import Path
from tqdm import tqdm

In [35]:
sample_image = next(TRAIN_IMAGES.glob("*.jpg"))
print(sample_image)

C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-train\VisDrone2019-DET-train\images\0000002_00005_d_0000014.jpg


In [38]:
import shutil

sample_image = next(TRAIN_IMAGES.glob("*.jpg"))

destination = TRAIN_OUT_IMAGES / sample_image.name

print("Source      :", sample_image)
print("Destination :", destination)

print("Source Exists      :", sample_image.exists())
print("Destination Folder :", TRAIN_OUT_IMAGES.exists())

shutil.copy2(str(sample_image), str(destination))

print("✅ Copy Successful")

Source      : C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-train\VisDrone2019-DET-train\images\0000002_00005_d_0000014.jpg
Destination : C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\processed\images\train\0000002_00005_d_0000014.jpg
Source Exists      : True
Destination Folder : True


FileNotFoundError: [WinError 3] The system cannot find the path specified

In [33]:
from pathlib import Path

# ==========================================================
# RAW DATASET
# ==========================================================

RAW_DATA = Path(r"C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET")

TRAIN_IMAGES = RAW_DATA / "VisDrone2019-DET-train" / "VisDrone2019-DET-train" / "images"
TRAIN_LABELS = RAW_DATA / "VisDrone2019-DET-train" / "VisDrone2019-DET-train" / "annotations"

VAL_IMAGES = RAW_DATA / "VisDrone2019-DET-val" / "VisDrone2019-DET-val" / "images"
VAL_LABELS = RAW_DATA / "VisDrone2019-DET-val" / "VisDrone2019-DET-val" / "annotations"

# ==========================================================
# PROCESSED DATASET
# ==========================================================

PROCESSED = Path(r"C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\processed")

TRAIN_OUT_IMAGES = PROCESSED / "images" / "train"
TRAIN_OUT_LABELS = PROCESSED / "labels" / "train"

VAL_OUT_IMAGES = PROCESSED / "images" / "val"
VAL_OUT_LABELS = PROCESSED / "labels" / "val"

In [36]:
label_path = TRAIN_LABELS / f"{sample_image.stem}.txt"

print(label_path)
print(label_path.exists())

C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-train\VisDrone2019-DET-train\annotations\0000002_00005_d_0000014.txt
True


In [34]:
print(TRAIN_OUT_IMAGES)
print(TRAIN_OUT_LABELS)
print(VAL_OUT_IMAGES)
print(VAL_OUT_LABELS)

print(type(TRAIN_OUT_IMAGES))
print(type(TRAIN_OUT_LABELS))
print(type(VAL_OUT_IMAGES))
print(type(VAL_OUT_LABELS))

C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\processed\images\train
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\processed\labels\train
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\processed\images\val
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\processed\labels\val
<class 'pathlib.WindowsPath'>
<class 'pathlib.WindowsPath'>
<class 'pathlib.WindowsPath'>
<class 'pathlib.WindowsPath'>


In [37]:
import cv2

image = cv2.imread(str(sample_image))

print(image is None)

False


In [11]:
folders = [
    TRAIN_OUT_IMAGES,
    TRAIN_OUT_LABELS,
    VAL_OUT_IMAGES,
    VAL_OUT_LABELS
]

for folder in folders:
    print(folder, type(folder))

C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\processed\images\train <class 'pathlib.WindowsPath'>
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\processed\labels\train <class 'pathlib.WindowsPath'>
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\processed\images\val <class 'pathlib.WindowsPath'>
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\processed\labels\val <class 'pathlib.WindowsPath'>


In [12]:
def visdrone_to_yolo(x, y, w, h, img_width, img_height):
    """
    Convert VisDrone bounding box format
    to YOLO bounding box format.
    """

    x_center = (x + w / 2) / img_width
    y_center = (y + h / 2) / img_height

    w = w / img_width
    h = h / img_height

    return x_center, y_center, w, h

In [13]:
sample_image = next(TRAIN_IMAGES.glob("*.jpg"))

sample_label = TRAIN_LABELS / f"{sample_image.stem}.txt"

print("Image :", sample_image.name)
print("Label :", sample_label.name)

Image : 0000002_00005_d_0000014.jpg
Label : 0000002_00005_d_0000014.txt


In [14]:
image = cv2.imread(str(sample_image))

img_height, img_width = image.shape[:2]

print(f"Image Width  : {img_width}")
print(f"Image Height : {img_height}")

Image Width  : 960
Image Height : 540


In [16]:
with open(sample_label, "r") as f:
    annotations = f.readlines()

print(f"Total Objects : {len(annotations)}")

print("\nFirst Five Objects\n")

for ann in annotations[:5]:
    print(ann.strip())

Total Objects : 88

First Five Objects

684,8,273,116,0,0,0,0
406,119,265,70,0,0,0,0
255,22,119,128,0,0,0,0
1,3,209,78,0,0,0,0
708,471,74,33,1,4,0,1


In [17]:
print("YOLO Labels\n")

for ann in annotations[:5]:

    values = ann.strip().split(",")

    x = int(values[0])
    y = int(values[1])
    w = int(values[2])
    h = int(values[3])

    score = int(values[4])
    class_id = int(values[5])

    # Ignore ignored regions
    if score == 0:
        continue

    # Convert class IDs from 1–10 to 0–9
    class_id -= 1

    x_center, y_center, w, h = visdrone_to_yolo(
        x, y, w, h,
        img_width,
        img_height
    )

    print(
        class_id,
        f"{x_center:.6f}",
        f"{y_center:.6f}",
        f"{w:.6f}",
        f"{h:.6f}"
    )

YOLO Labels

3 0.776042 0.902778 0.077083 0.061111


In [42]:
import shutil
import traceback
from tqdm import tqdm

def process_dataset(image_dir, label_dir, output_image_dir, output_label_dir):

    # Create output folders if they don't exist
    output_image_dir.mkdir(parents=True, exist_ok=True)
    output_label_dir.mkdir(parents=True, exist_ok=True)

    image_paths = sorted(image_dir.glob("*.jpg"))

    processed = 0
    skipped = 0

    print(f"\nFound {len(image_paths)} images.\n")

    for image_path in tqdm(image_paths, desc="Processing images"):

        try:

            # --------------------------------------------------
            # Label path
            # --------------------------------------------------
            label_path = label_dir / f"{image_path.stem}.txt"

            if not label_path.exists():
                print(f"\n❌ Label Missing : {label_path}")
                skipped += 1
                continue

            # --------------------------------------------------
            # Read Image
            # --------------------------------------------------
            image = cv2.imread(str(image_path))

            if image is None:
                print(f"\n❌ Unable to read image : {image_path}")
                skipped += 1
                continue

            img_height, img_width = image.shape[:2]

            # --------------------------------------------------
            # Copy Image
            # --------------------------------------------------
            destination_image = output_image_dir / image_path.name

            destination_image.parent.mkdir(parents=True, exist_ok=True)

            shutil.copy2(
                str(image_path.resolve()),
                str(destination_image.resolve())
            )

            # --------------------------------------------------
            # Read Annotation
            # --------------------------------------------------
            with open(label_path, "r") as f:
                annotations = f.readlines()

            yolo_annotations = []

            # --------------------------------------------------
            # Convert Every Bounding Box
            # --------------------------------------------------
            for ann in annotations:

                values = ann.strip().split(",")

                x = int(values[0])
                y = int(values[1])
                w = int(values[2])
                h = int(values[3])

                score = int(values[4])
                class_id = int(values[5])

                # Ignore ignored regions
                if score == 0:
                    continue

                # Convert VisDrone IDs (1-10) → YOLO IDs (0-9)
                class_id -= 1

                x_center, y_center, bw, bh = visdrone_to_yolo(
                    x,
                    y,
                    w,
                    h,
                    img_width,
                    img_height
                )

                yolo_annotations.append(
                    f"{class_id} "
                    f"{x_center:.6f} "
                    f"{y_center:.6f} "
                    f"{bw:.6f} "
                    f"{bh:.6f}"
                )

            # --------------------------------------------------
            # Save YOLO Label
            # --------------------------------------------------
            output_label = output_label_dir / f"{image_path.stem}.txt"

            output_label.parent.mkdir(parents=True, exist_ok=True)

            with open(output_label.resolve(), "w") as f:
                f.write("\n".join(yolo_annotations))

            processed += 1

        except Exception:

            print("\n====================================")
            print(f"❌ Error Processing : {image_path.name}")
            print("====================================")

            traceback.print_exc()

            print("\nImage Path:")
            print(image_path)

            print("\nLabel Path:")
            print(label_path)

            print("\nDestination Image:")
            print(destination_image)

            print("\nDestination Label:")
            print(output_label)

            skipped += 1

    print("\n===================================")
    print("Processing Finished")
    print("===================================")

    print(f"Processed : {processed}")
    print(f"Skipped   : {skipped}")

In [47]:
test_images = sorted(TRAIN_IMAGES.glob("*.jpg"))[:5]

process_dataset(
    image_dir=TRAIN_IMAGES,
    label_dir=TRAIN_LABELS,
    output_image_dir=TRAIN_OUT_IMAGES,
    output_label_dir=TRAIN_OUT_LABELS
)


Found 6471 images.



Processing images: 100%|██████████| 6471/6471 [03:44<00:00, 28.80it/s]


Processing Finished
Processed : 6471
Skipped   : 0


In [44]:
import os

print("Exists:", os.path.exists(TRAIN_OUT_IMAGES))
print("Is Directory:", os.path.isdir(TRAIN_OUT_IMAGES))
print("Is File:", os.path.isfile(TRAIN_OUT_IMAGES))

Exists: True
Is Directory: False
Is File: True


In [45]:
import shutil
from pathlib import Path

PROCESSED = Path(r"C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\processed")

# Remove old processed folder if it exists
if PROCESSED.exists():
    shutil.rmtree(PROCESSED)

# Create correct folder structure
(PROCESSED / "images" / "train").mkdir(parents=True, exist_ok=True)
(PROCESSED / "images" / "val").mkdir(parents=True, exist_ok=True)

(PROCESSED / "labels" / "train").mkdir(parents=True, exist_ok=True)
(PROCESSED / "labels" / "val").mkdir(parents=True, exist_ok=True)

print("✅ Folder structure recreated successfully!")

✅ Folder structure recreated successfully!


In [46]:
import os

print(os.path.isdir(PROCESSED / "images" / "train"))
print(os.path.isdir(PROCESSED / "images" / "val"))

print(os.path.isdir(PROCESSED / "labels" / "train"))
print(os.path.isdir(PROCESSED / "labels" / "val"))

True
True
True
True


In [20]:
print(RAW_DATA)

C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET


In [21]:
for item in RAW_DATA.iterdir():
    print(item)

C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone.yaml
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-test-challenge
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-test-dev
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-train
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-val


In [22]:
for item in (RAW_DATA / "VisDrone2019-DET-train").iterdir():
    print(item)

C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-train\VisDrone2019-DET-train


In [23]:
print(TRAIN_OUT_IMAGES.exists())
print(TRAIN_OUT_LABELS.exists())

print(VAL_OUT_IMAGES.exists())
print(VAL_OUT_LABELS.exists())

True
True
True
True


In [30]:
from pathlib import Path

RAW_DATA = Path(r"C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET")

for path in RAW_DATA.rglob("*"):
    if path.is_dir():
        print(path)

C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-test-challenge
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-test-dev
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-train
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-val
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-test-challenge\VisDrone2019-DET-test-challenge
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-test-dev\VisDrone2019-DET-test-dev
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-train\VisDrone2019-DET-train
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-val\VisDrone2019-DET-val
C:\Users\nanda\OneDrive\Docume

In [39]:
print("Training Images :", len(list(TRAIN_OUT_IMAGES.glob("*.jpg"))))
print("Training Labels :", len(list(TRAIN_OUT_LABELS.glob("*.txt"))))

print()

print("Validation Images :", len(list(VAL_OUT_IMAGES.glob("*.jpg"))))
print("Validation Labels :", len(list(VAL_OUT_LABELS.glob("*.txt"))))

Training Images : 0
Training Labels : 0

Validation Images : 0
Validation Labels : 0


In [51]:
print("========== TRAIN ==========")
print("Images :", len(list(TRAIN_OUT_IMAGES.glob("*.jpg"))))
print("Labels :", len(list(TRAIN_OUT_LABELS.glob("*.txt"))))

print("\n========== VALIDATION ==========")
print("Images :", len(list(VAL_OUT_IMAGES.glob("*.jpg"))))
print("Labels :", len(list(VAL_OUT_LABELS.glob("*.txt"))))

========== TRAIN ==========
Images : 6471
Labels : 6471

========== VALIDATION ==========
Images : 548
Labels : 548


In [49]:
print(VAL_IMAGES)
print(VAL_LABELS)

print("\nValidation Images:", len(list(VAL_IMAGES.glob("*.jpg"))))
print("Validation Labels:", len(list(VAL_LABELS.glob("*.txt"))))

C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-val\VisDrone2019-DET-val\images
C:\Users\nanda\OneDrive\Documents\UAV - Computer Vision\data\raw\VisDrone-2019-DET\VisDrone2019-DET-val\VisDrone2019-DET-val\annotations

Validation Images: 548
Validation Labels: 548


In [50]:
process_dataset(
    image_dir=VAL_IMAGES,
    label_dir=VAL_LABELS,
    output_image_dir=VAL_OUT_IMAGES,
    output_label_dir=VAL_OUT_LABELS
)


Found 548 images.



Processing images: 100%|██████████| 548/548 [00:17<00:00, 31.18it/s]


Processing Finished
Processed : 548
Skipped   : 0
